# 01 — Data Cleaning and Network Construction

This notebook loads one month of flight data, cleans airport coordinates and
constructs domestic flight networks for four countries:
**USA, China, United Kingdom and Australia**.

Each country's domestic flights are modelled as an undirected weighted graph
where airports are nodes and flights are edges.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
from src.data_preprocessing import (
    load_data, check_missing_codes, correct_coordinates,
    get_domestic_flights, merge_coordinates
)
from src.visualisation import plot_network_map

## Load and inspect data

In [ ]:
airports, flights = load_data('../data/Airports.csv', '../data/Flight Data.xlsx')

print(f'Airports: {airports.shape}')
print(f'Flights:  {flights.shape}')
print(f'\nMissing values (airports):\n{airports.isna().sum()}')
print(f'\nCoordinate summary:\n{airports[["Lat", "Lon"]].describe()}')

In [ ]:
missing = check_missing_codes(airports, flights)
print(f"Airport codes in flights but not in airports table: {missing['total_missing']}")

## Coordinate correction

Some airports have incorrect coordinates in the raw data:
- **Australia**: airports recorded with positive latitude (northern hemisphere) are corrected to negative.
- **USA**: eight airports with known incorrect positions are manually fixed.

Only coordinates change — all network connections are preserved.

In [ ]:
airports = correct_coordinates(airports)

corrected_count = airports['is_corrected'].sum()
print(f'Total airports corrected: {corrected_count}')

## Network statistics

In [ ]:
import pandas as pd

countries = ['USA', 'China', 'United Kingdom', 'Australia']
stats = []

for c in countries:
    df_c = get_domestic_flights(flights, c)
    nodes = set(df_c['Source']).union(set(df_c['Target']))
    stats.append({'Country': c, 'Domestic flights': len(df_c), 'Unique airports': len(nodes)})

pd.DataFrame(stats)

## Geospatial visualisation (cleaned coordinates)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    dom = get_domestic_flights(flights, country)
    merged = merge_coordinates(dom, airports)
    plot_network_map(merged, airports, country, ax)

plt.tight_layout()
plt.savefig('../figures/cleaned_network_maps.png', dpi=150, bbox_inches='tight')
plt.show()